In [79]:
import pandas as pd
import numpy as np

In [80]:
mca_df = pd.read_csv('../data/auxiliary/final_mca_soc_code.csv')
mca_df["Job Sector"] = (
    mca_df["Job Sector"].replace(
        {
            "Arts, entertainment and recreation": "Arts, Entertainment and Recreation",
            "Professional, Scientific And Technical Activities" : "Professional, Scientific and Technical Activities"
        }
    )
)

ai_measurements_df = pd.read_csv('../data/auxiliary/stage_3_archetypes.csv')

In [81]:
mca_AI_df = pd.read_csv('../data/auxiliary/final_mca_with_ai_measurements.csv')

In [82]:
excel_tables = {}

# 1. Creating Occupation Master

| Column name          | Value or calculation                                                                           |
| -------------------- | ---------------------------------------------------------------------------------------------- |
| `SOC_code`           | Occupation code used in the existing analysis.                                                 |
| `SOC_title`          | Corresponding occupation title.                                                                |
| `Sector`             | Sector assigned in the existing analysis.                                                      |
| `AI_potential_score` | Existing value used for the potential axis.                                                    |
| `AI_observed_score`  | Existing value used for the observed-use axis.                                                 |
| `AI_archetype`       | Existing classification in the current run.                                                    |
| `MCA_entry_count`    | Number of distinct MCA career entries mapped to this SOC code under the existing mapping rule. |
| `In_MCA`             | Yes if `MCA_entry_count` > 0; otherwise No.                                                    |
| `Potential_imputed`  | Yes, No, or Unknown.                                                                           |
| `Observed_imputed`   | Yes, No, or Unknown.                                                                           |

In [83]:
mca_entry_count_dict = mca_AI_df.groupby('SOC Code')['Job Title'].count().to_dict()

Archetype C-AIOE Percentile - Observed Exposure 80

In [84]:
rename_columns = {
    'O*NET-SOC Code': 'SOC_code',
    'Title': 'SOC_title',
    'Major Group': 'Sector',
    'C-AIOE': 'AI_potential_score',
    'Observed Exposure': 'AI_observed_score',
    'Archetype C-AIOE Percentile 50': 'Archetype C-AIOE Percentile - Observed Exposure 50',
    'Archetype C-AIOE Percentile 60': 'Archetype C-AIOE Percentile - Observed Exposure 60',
    'Archetype C-AIOE Percentile 75': 'Archetype C-AIOE Percentile - Observed Exposure 75',
    #'Archetype C-AIOE Percentile 80': 'Archetype C-AIOE Percentile 80'
}
occupation_master = ai_measurements_df.rename(columns = rename_columns)[rename_columns.values()]
occupation_master['MCA_entry_count'] = occupation_master['SOC_code'].apply(lambda x : mca_entry_count_dict.get(x, 0))
occupation_master['In_MCA'] = occupation_master.apply(lambda row : 'No' if row['MCA_entry_count'] == 0 else 'Yes', axis=1)
occupation_master['Potential_imputed'] = (
    occupation_master['AI_potential_score']
    .isna()
    .map({True: 'Yes', False: 'No'})
)
occupation_master['Observed_imputed'] = (
    occupation_master['AI_observed_score']
    .isna()
    .map({True: 'Yes', False: 'No'})
)

In [85]:
occupation_master[occupation_master['In_MCA'] == 'No'].groupby('Sector')['SOC_code'].count().divide(
    occupation_master.groupby('Sector')['SOC_code'].count()
).sort_values()

Sector
Arts, Design, Entertainment, Sports, and Media    0.212121
Computer and Mathematical                         0.263158
Healthcare Support                                0.363636
Life, Physical, and Social Science                0.368421
Transportation and Material Moving                0.368421
Construction and Extraction                       0.428571
Architecture and Engineering                      0.451613
Office and Administrative Support                 0.468085
Production                                        0.487179
Sales and Related                                 0.500000
Building and Grounds Cleaning and Maintenance     0.500000
Community and Social Service                      0.500000
Food Preparation and Serving Related              0.538462
Installation, Maintenance, and Repair             0.571429
Legal                                             0.571429
Business and Financial Operations                 0.576923
Personal Care and Service                        

In [86]:
excel_tables['Occupation_Master'] = occupation_master

# 2. Archetype Comparison

| Column name                | Value or calculation                                                            |
| -------------------------- | ------------------------------------------------------------------------------- |
| `AI_archetype`             | Existing archetype name.                                                        |
| `SOC_count`                | Number of comparison-sample SOC occupations in the archetype.                   |
| `SOC_share_pct`            | `SOC_count` ÷ total classified comparison-sample SOC occupations × 100.         |
| `MCA_entry_count`          | Number of distinct classified MCA entries in the archetype.                     |
| `MCA_entry_share_pct`      | `MCA_entry_count` ÷ total classified MCA entries × 100.                         |
| `MCA_unique_SOC_count`     | Number of distinct classified MCA-matched SOC codes in the archetype.           |
| `MCA_unique_SOC_share_pct` | `MCA_unique_SOC_count` ÷ total distinct classified MCA-matched SOC codes × 100. |
| `Entry_gap_pp`             | `MCA_entry_share_pct` − `SOC_share_pct`.                                        |
| `Unique_SOC_gap_pp`        | `MCA_unique_SOC_share_pct` − `SOC_share_pct`.                                   |

In [87]:
thresholds = [50, 60, 75]
comparison_dfs = []

for threshold in thresholds:
    archetype_percentile_threshold = (
        f'Archetype C-AIOE Percentile - Observed Exposure {threshold}'
    )

    # SOC
    soc_entry = ai_measurements_df.groupby(
        archetype_percentile_threshold
    )['O*NET-SOC Code'].count()

    soc_entry_pct = (soc_entry / soc_entry.sum()) * 100

    # MCA Entry
    mca_entry = mca_AI_df.groupby(
        archetype_percentile_threshold
    )['SOC Code'].count()

    mca_entry_pct = (mca_entry / mca_entry.sum()) * 100

    # MCA Unique SOC
    mca_unique_entry = (
        mca_AI_df
        .drop_duplicates(subset=['SOC Code'])
        .groupby(archetype_percentile_threshold)['SOC Code']
        .count()
    )

    mca_unique_entry_pct = (
        mca_unique_entry / mca_unique_entry.sum()
    ) * 100

    # Combine metrics
    entries = [
        soc_entry,
        soc_entry_pct,
        mca_entry,
        mca_entry_pct,
        mca_unique_entry,
        mca_unique_entry_pct
    ]

    archetype_comparison_df = pd.concat(
        entries, axis=1
    ).reset_index()

    archetype_comparison_df.columns = [
        'AI_archetype',
        'SOC_count',
        'SOC_share_pct',
        'MCA_entry_count',
        'MCA_entry_share_pct',
        'MCA_unique_SOC_count',
        'MCA_unique_SOC_share_pct'
    ]

    # Calculate percentage-point gaps
    archetype_comparison_df['Entry_gap_pp'] = (
        archetype_comparison_df['MCA_entry_share_pct']
        - archetype_comparison_df['SOC_share_pct']
    )

    archetype_comparison_df['Unique_SOC_gap_pp'] = (
        archetype_comparison_df['MCA_unique_SOC_share_pct']
        - archetype_comparison_df['SOC_share_pct']
    )

    # Track the threshold
    archetype_comparison_df.insert(0, 'Threshold', threshold)

    comparison_dfs.append(archetype_comparison_df)

# Combine all thresholds into one table
archetype_comparison_all_df = pd.concat(
    comparison_dfs,
    ignore_index=True
)

archetype_comparison_all_df

,Threshold,AI_archetype,SOC_count,SOC_share_pct,MCA_entry_count,MCA_entry_share_pct,MCA_unique_SOC_count,MCA_unique_SOC_share_pct,Entry_gap_pp,Unique_SOC_gap_pp
0,50,AI-Enabled Expansion,31,5.772812,40,4.694836,13,5.038760,-1.077976,-0.734052
1,50,Active Transformation,127,23.649907,314,36.854460,75,29.069767,13.204553,5.419861
2,50,Latent Transformation,142,26.443203,154,18.075117,61,23.643411,-8.368086,-2.799792
3,50,Limited Current Transformation,237,44.134078,344,40.375587,109,42.248062,-3.758491,-1.886016
4,60,AI-Enabled Expansion,19,3.538175,33,3.873239,8,3.100775,0.335064,-0.437400
5,60,Active Transformation,108,20.111732,298,34.976526,66,25.581395,14.864794,5.469664
6,60,Latent Transformation,161,29.981378,170,19.953052,70,27.131783,-10.028326,-2.849595
7,60,Limited Current Transformation,249,46.368715,351,41.197183,114,44.186047,-5.171532,-2.182669
8,75,AI-Enabled Expansion,8,1.489758,28,3.286385,5,1.937984,1.796627,0.448227
9,75,Active Transformation,71,13.221601,209,24.530516,48,18.604651,11.308915,5.383050


In [88]:
excel_tables['Archetype_Comparison'] = archetype_comparison_all_df

# 3. Sector Comparison

| Column                            | Definition                                                                |
| --------------------------------- | ------------------------------------------------------------------------- |
| `Sector`                          | Existing sector name. Use `Unknown` for unassigned sectors and flag them. |
| `SOC_total`                       | All classified comparison-sample occupations in this sector.              |
| `SOC_active_count`                | Comparison-sample occupations classified as Active Transformation.        |
| `SOC_active_pct`                  | `SOC_active_count` ÷ `SOC_total` × 100.                                   |
| `MCA_entries_total`               | All distinct classified MCA entries in this sector.                       |
| `MCA_entries_active_count`        | MCA entries classified as Active Transformation.                          |
| `MCA_entries_active_pct`          | `MCA_entries_active_count` ÷ `MCA_entries_total` × 100.                   |
| `MCA_unique_SOC_total`            | Distinct classified MCA-matched SOC codes in this sector.                 |
| `MCA_unique_SOC_active_count`     | Distinct MCA-matched SOC codes classified as Active Transformation.       |
| `MCA_unique_SOC_active_pct`       | `MCA_unique_SOC_active_count` ÷ `MCA_unique_SOC_total` × 100.             |
| `Within_sector_gap_pp`            | `MCA_unique_SOC_active_pct` − `SOC_active_pct`.                           |
| `Share_of_MCA_active_entries_pct` | `MCA_entries_active_count` ÷ all MCA Active Transformation entries × 100. |

 - Observed Exposure

In [89]:
active_label = 'Active Transformation'
ai_archetype_threshold ='Archetype C-AIOE Percentile - Observed Exposure 50'
# SOC comparison sample
soc_summary = (
    ai_measurements_df
    .groupby('Major Group')
    .agg(
        SOC_total=('O*NET-SOC Code', 'count'),
        SOC_active_count=(
            ai_archetype_threshold,
            lambda x: (x == active_label).sum()
        )
    )
)

soc_summary['SOC_active_pct'] = (
    soc_summary['SOC_active_count']
    / soc_summary['SOC_total'] * 100
)

# MCA entries
mca_summary = (
    mca_AI_df
    .groupby('Major Group')
    .agg(
        MCA_entries_total=('SOC Code', 'count'),
        MCA_entries_active_count=(
            ai_archetype_threshold,
            lambda x: (x == active_label).sum()
        )
    )
)

mca_summary['MCA_entries_active_pct'] = (
    mca_summary['MCA_entries_active_count']
    / mca_summary['MCA_entries_total'] * 100
)

# MCA unique SOC codes
mca_unique_df = mca_AI_df.drop_duplicates(
    subset=['SOC Code']
)

mca_unique_summary = (
    mca_unique_df
    .groupby('Major Group')
    .agg(
        MCA_unique_SOC_total=('SOC Code', 'count'),
        MCA_unique_SOC_active_count=(
            ai_archetype_threshold,
            lambda x: (x == active_label).sum()
        )
    )
)

mca_unique_summary['MCA_unique_SOC_active_pct'] = (
    mca_unique_summary['MCA_unique_SOC_active_count']
    / mca_unique_summary['MCA_unique_SOC_total'] * 100
)

# Combine summaries
sector_comparison_df = (
    soc_summary
    .join(mca_summary, how='outer')
    .join(mca_unique_summary, how='outer')
    .reset_index()
    .rename(columns={'Major Group': 'Sector'})
)

# Within-sector percentage-point gap
sector_comparison_df['Within_sector_gap_pp'] = (
    sector_comparison_df['MCA_unique_SOC_active_pct']
    - sector_comparison_df['SOC_active_pct']
)

# Share of all MCA active entries
total_mca_active_entries = (
    sector_comparison_df['MCA_entries_active_count'].sum()
)

sector_comparison_df['Share_of_MCA_active_entries_pct'] = (
    sector_comparison_df['MCA_entries_active_count']
    / total_mca_active_entries * 100
)

sector_comparison_df = sector_comparison_df.sort_values(
    'Sector'
).reset_index(drop=True)

sector_comparison_df.sort_values(by=['MCA_entries_active_count', 'Sector'], ascending=[False, True], inplace=True)

In [90]:
excel_tables['Sector_Comparison'] = sector_comparison_df

# 4. Six Occupations

| Column name               | Value or calculation                                                                                                                               |
| ------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| `SOC_code`                | Exact SOC code.                                                                                                                                    |
| `SOC_title`               | Exact SOC occupation title.                                                                                                                        |
| `AI_potential_score`      | AI potential score from the current run.                                                                                                           |
| `AI_observed_score`       | AI observed score from the current run.                                                                                                            |
| `Current_archetype`       | Classification in the current run.                                                                                                                 |
| `MCA_entry_count`         | Number of distinct mapped MCA entries.                                                                                                             |
| `MCA_titles`              | Mapped career titles separated by semicolons. If none, enter `None`.                                                                               |
| `Archetype_[run_name]`    | One separate column per existing comparable run. Enter that run's classification, or `Not available`.                                              |
| `Expansion_runs`          | Number of included runs classifying the occupation as `AI-Enabled Expansion`.                                                                      |
| `Available_runs`          | Number of included runs with a classification for that occupation.                                                                                 |
| `Expansion_frequency_pct` | `Expansion_runs ÷ Available_runs × 100`. Enter `N/A` if `Available_runs = 0`.                                                                      |
| `Data_flag`               | List all applicable flags: missing potential score, missing observed score, imputation, uncertain mapping, or `None identified` if no flags apply. |

In [91]:
six_occupations = ['First-Line Supervisors of Retail Sales Workers',
 'Photographers',
 'Food Science Technicians',
 'Computer User Support Specialists',
 'Dietetic Technicians',
 'Art, Drama, and Music Teachers, Postsecondary',
 'Curators',
 'Radiologists',
 'Social and Community Service Managers',
 'Education Teachers, Postsecondary',
 'Secondary School Teachers, Except Special and Career/Technical Education',
 'Health Information Technologists and Medical Registrars',
 'First-Line Supervisors of Non-Retail Sales Workers',
 'Health Specialties Teachers, Postsecondary',
 'Middle School Teachers, Except Special and Career/Technical Education',
 'Chemistry Teachers, Postsecondary',
 'Nursing Instructors and Teachers, Postsecondary',
 'Biological Science Teachers, Postsecondary',
 'Judges, Magistrate Judges, and Magistrates']


columns_to_get = [
    'SOC_code',
    'SOC_title',
    'AI_potential_score',
    'AI_observed_score',
    'Archetype C-AIOE Percentile - Observed Exposure 50',
    'Archetype C-AIOE Percentile - Observed Exposure 60',
    'Archetype C-AIOE Percentile - Observed Exposure 75'
]

archetype_columns = [
    'Archetype C-AIOE Percentile - Observed Exposure 50',
    'Archetype C-AIOE Percentile - Observed Exposure 60',
    'Archetype C-AIOE Percentile - Observed Exposure 75'
]



# Get the six occupations and their current scores and classifications.
six_occupations_df = (
    occupation_master.loc[
        occupation_master['SOC_title'].isin(six_occupations),
        columns_to_get
    ]
    .copy()
)

six_occupations_df['Current_archetype'] = (
    six_occupations_df['Archetype C-AIOE Percentile - Observed Exposure 50']
)

# Get mapped MCA entries for the six occupations.
mca_filtered_df = mca_AI_df[
    mca_AI_df['SOC Code'].isin(six_occupations_df['SOC_code'])
].copy()

# Count distinct mapped MCA entries and collect their titles.
mca_summary_df = (
    mca_filtered_df.groupby('SOC Code', as_index=False)
    .agg(
        MCA_entry_count=('Job Title', 'nunique'),
        MCA_titles=(
            'Job Title',
            lambda x: '; '.join(
                x.dropna().astype(str).unique()
            ) if x.notna().any() else 'None'
        )
    )
    .rename(columns={'SOC Code': 'SOC_code'})
)

# Rename each run's classification column.
run_name_mapping = {
    column: column.replace('Archetype ', 'Archetype_').replace(
        ' ', '_'
    )
    for column in archetype_columns
}

archetype_summary_df = six_occupations_df[
    ['SOC_code', 'SOC_title', 'AI_potential_score',
     'AI_observed_score', 'Current_archetype'] + archetype_columns
].rename(columns=run_name_mapping)

# Count expansion classifications and available classifications.
expansion_runs = (
    six_occupations_df[archetype_columns]
    .eq('AI-Enabled Expansion')
    .sum(axis=1)
)

available_runs = (
    six_occupations_df[archetype_columns]
    .ne('Unknown')
    .sum(axis=1)
)

# Exclude unavailable classifications from the denominator.
expansion_frequency_pct = (
    expansion_runs.div(available_runs.replace(0, np.nan)) * 100
)

# Combine the occupation, MCA, and run summaries.
six_occupations_comparison_df = (
    archetype_summary_df
    .merge(mca_summary_df, on='SOC_code', how='left')
    .assign(
        Expansion_runs=expansion_runs.to_numpy(),
        Available_runs=available_runs.to_numpy(),
        Expansion_frequency_pct=expansion_frequency_pct.to_numpy()
    )
)

# Fill occupations without mapped MCA entries.
six_occupations_comparison_df['MCA_entry_count'] = (
    six_occupations_comparison_df['MCA_entry_count']
    .fillna(0)
    .astype(int)
)

six_occupations_comparison_df['MCA_titles'] = (
    six_occupations_comparison_df['MCA_titles'].fillna('None')
)

# Replace Unknown with Not available in individual run columns.
run_columns = list(run_name_mapping.values())

six_occupations_comparison_df[run_columns] = (
    six_occupations_comparison_df[run_columns]
    .fillna('Not available')
    .replace('Unknown', 'Not available')
)

# Use N/A when no comparable runs are available.
six_occupations_comparison_df['Expansion_frequency_pct'] = (
    six_occupations_comparison_df['Expansion_frequency_pct']
    .round(2)
    .astype(object)
)

six_occupations_comparison_df.loc[
    six_occupations_comparison_df['Available_runs'].eq(0),
    'Expansion_frequency_pct'
] = 'N/A'

# Add data-quality flags.
# Adjust these source column names to match your actual data.
def get_data_flags(row: pd.Series) -> str:
    """Return applicable data-quality flags for an occupation."""
    flags = []

    if pd.isna(row['AI_potential_score']):
        flags.append('Missing potential score')

    if pd.isna(row['AI_observed_score']):
        flags.append('Missing observed score')

    # Add your imputation and uncertain-mapping checks here
    # if those indicators are available in your source data.

    return '; '.join(flags) if flags else 'None identified'


six_occupations_comparison_df['Data_flag'] = (
    six_occupations_comparison_df.apply(get_data_flags, axis=1)
)

# Arrange columns in the requested order.
six_occupations_comparison_df = six_occupations_comparison_df[
    [
        'SOC_code',
        'SOC_title',
        'AI_potential_score',
        'AI_observed_score',
        'Current_archetype',
        'MCA_entry_count',
        'MCA_titles',
        *run_columns,
        'Expansion_runs',
        'Available_runs',
        'Expansion_frequency_pct',
        'Data_flag'
    ]
]
excel_tables['Six_Occupations'] = six_occupations_comparison_df

In [92]:
ai_measurements_df[ai_measurements_df['O*NET-SOC Code'] == '27-1013.00']

,O*NET-SOC Code,Title,Major Group,AIOE,Complementarity,C-AIOE,Observed Exposure,LLM Rating Beta,Human Rating Beta,Archetype C-AIOE Percentile - Observed Exposure 50,...,Archetype C-AIOE Percentile - Observed Exposure 80,Augmentation Percent,Automation Percent,Directive Percent,Feedback Loop Percent,Learning Percent,Task Iteration Percent,Validation Percent,None Percent,Sub-Archetype
38,27-1013.00,"Fine Artists, Including Painters, Sculptors, a...","Arts, Design, Entertainment, Sports, and Media",5.775394,45.9375,5.30855,0.3565,0.425,0.3125,Active Transformation,...,Active Transformation,55.19,44.81,38.37,5.91,3.71,49.86,0.97,1.18,AI-Powered


In [93]:
six_occupations_comparison_df[six_occupations_comparison_df['Expansion_runs'] == 2]

,SOC_code,SOC_title,AI_potential_score,AI_observed_score,Current_archetype,MCA_entry_count,MCA_titles,Archetype_C-AIOE_Percentile_-_Observed_Exposure_50,Archetype_C-AIOE_Percentile_-_Observed_Exposure_60,Archetype_C-AIOE_Percentile_-_Observed_Exposure_75,Expansion_runs,Available_runs,Expansion_frequency_pct,Data_flag
0,41-1011.00,First-Line Supervisors of Retail Sales Workers,4.669898,0.2627,AI-Enabled Expansion,2,Store Supervisor Trainee; Online Store Executi...,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
1,27-4021.00,Photographers,4.563433,0.1950,AI-Enabled Expansion,2,Photographer; Professional Photographer,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
2,19-4013.00,Food Science Technicians,4.693805,0.2038,AI-Enabled Expansion,1,Master Tagger,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
4,29-2051.00,Dietetic Technicians,4.393992,0.2446,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
5,25-1121.00,"Art, Drama, and Music Teachers, Postsecondary",4.619587,0.2142,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
7,29-1224.00,Radiologists,4.475451,0.1990,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
8,11-9151.00,Social and Community Service Managers,4.667980,0.1808,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
9,25-1081.00,"Education Teachers, Postsecondary",4.666652,0.2621,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
12,41-1012.00,First-Line Supervisors of Non-Retail Sales Wor...,4.497926,0.2295,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified
15,25-1052.00,"Chemistry Teachers, Postsecondary",4.446115,0.2185,AI-Enabled Expansion,0,None,AI-Enabled Expansion,AI-Enabled Expansion,Limited Current Transformation,2,3,66.67,None identified


# 5. Checks and submission

| Check                                  | Expected                                                                                                    | Actual | Pass/Flag | Notes |
| -------------------------------------- | ----------------------------------------------------------------------------------------------------------- | ------ | --------- | ----- |
| Comparison sample count                | 537 total occupations                                                                                       |        |           |       |
| Comparison Active Transformation count | 139 occupations                                                                                             |        |           |       |
| MCA classified entry count             | 852 entries                                                                                                 |        |           |       |
| MCA Active Transformation count        | 322 entries                                                                                                 |        |           |       |
| Archetype count totals                 | Each count column sums to its declared denominator                                                          |        |           |       |
| Archetype share totals                 | Each share column sums to 100%, within a rounding tolerance of 0.1 × number of archetypes percentage points |        |           |       |
| Sector count totals                    | Each total agrees with its corresponding archetype-table count                                              |        |           |       |
| Unique master codes                    | Zero duplicate SOC codes in `Occupation_Master`                                                             |        |           |       |
| Common thresholds                      | Identical definitions, thresholds, transformations, and boundary rules across the two samples               |        |           |       |
| Six occupation rows                    | Exactly six requested occupations, with no duplicates                                                       |        |           |       |
| Expansion run counts                   | `Expansion_runs` ≤ `Available_runs`; all counted runs appear in the run key                                 |        |           |       |

In [94]:
537 == occupation_master.SOC_code.nunique()

True

In [95]:
139 == occupation_master['Archetype C-AIOE Percentile - Observed Exposure 50'].eq('Active Transformation').sum()

np.False_

In [96]:
852 == mca_AI_df['Archetype C-AIOE Percentile - Observed Exposure 50'].dropna().shape[0]

True

In [97]:
322 == mca_AI_df['Archetype C-AIOE Percentile - Observed Exposure 50'].eq('Active Transformation').sum()

np.False_

In [98]:
share_totals = (
    archetype_comparison_all_df
    .groupby('Threshold')[[
        'SOC_share_pct',
        'MCA_entry_share_pct',
        'MCA_unique_SOC_share_pct'
    ]]
    .sum()
)

checks = np.isclose(share_totals, 100, atol=0.1)

checks.all()

np.True_

In [99]:
active_archetype_comparison_all_df = archetype_comparison_all_df[
    archetype_comparison_all_df['AI_archetype'] == 'Active Transformation'] 

active_archetype_totals = (
    active_archetype_comparison_all_df
    .groupby('Threshold')[[
        'SOC_count',
        'MCA_entry_count',
        'MCA_unique_SOC_count'
    ]]
    .sum()
    .loc[50]
)

sector_totals = sector_comparison_df[[
    'SOC_active_count',
    'MCA_entries_active_count',
    'MCA_unique_SOC_active_count'
]].sum()

sector_archetype_totals = pd.DataFrame({
    'Archetype_Total': active_archetype_totals.to_numpy(),
    'Sector_Total': sector_totals.to_numpy(),
}, index=[
    'SOC_count',
    'MCA_entry_count',
    'MCA_unique_SOC_count'
])

sector_archetype_totals['Is_equal'] = (
    sector_archetype_totals['Archetype_Total']
    == sector_archetype_totals['Sector_Total']
)

sector_archetype_totals['Is_equal'].all()

np.True_

In [100]:
occupation_master.drop_duplicates(subset='SOC_code').shape == occupation_master.shape

True

In [101]:
(six_occupations_comparison_df['Expansion_runs'] <= six_occupations_comparison_df['Available_runs']).all()

np.True_

In [102]:
checks_df = pd.DataFrame([
    {
        'Check': 'Comparison sample count',
        'Expected': '537 total occupations',
        'Actual': occupation_master['SOC_code'].nunique(),
        'Pass/Flag': (
            'Pass' if occupation_master['SOC_code'].nunique() == 537
            else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'Comparison Active Transformation count',
        'Expected': '139 occupations',
        'Actual': occupation_master[
            'Archetype C-AIOE Percentile - Observed Exposure 50'
        ].eq('Active Transformation').sum(),
        'Pass/Flag': (
            'Pass' if occupation_master[
                'Archetype C-AIOE Percentile - Observed Exposure 50'
            ].eq('Active Transformation').sum() == 139
            else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'MCA classified entry count',
        'Expected': '852 entries',
        'Actual': mca_AI_df[
            'Archetype C-AIOE Percentile - Observed Exposure 50'
        ].notna().sum(),
        'Pass/Flag': (
            'Pass' if mca_AI_df[
                'Archetype C-AIOE Percentile - Observed Exposure 50'
            ].notna().sum() == 852
            else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'MCA Active Transformation count',
        'Expected': '322 entries',
        'Actual': mca_AI_df[
            'Archetype C-AIOE Percentile - Observed Exposure 50'
        ].eq('Active Transformation').sum(),
        'Pass/Flag': (
            'Pass' if mca_AI_df[
                'Archetype C-AIOE Percentile - Observed Exposure 50'
            ].eq('Active Transformation').sum() == 322
            else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'Archetype count totals',
        'Expected': 'Each count column sums to its declared denominator',
        'Actual': (
            'Pass' if (
                archetype_comparison_all_df
                .groupby('Threshold')[[
                    'SOC_count',
                    'MCA_entry_count',
                    'MCA_unique_SOC_count'
                ]].sum().eq(
                    archetype_comparison_all_df.groupby('Threshold')[
                        'SOC_count'
                    ].sum().to_numpy()[:, None]
                ).all().all()
            ) else 'See Notes'
        ),
        'Pass/Flag': 'Pass' if (
            archetype_comparison_all_df.groupby('Threshold')[
                'SOC_count'
            ].sum().eq(537).all()
            and archetype_comparison_all_df.groupby('Threshold')[
                'MCA_entry_count'
            ].sum().eq(852).all()
        ) else 'Flag',
        'Notes': 'Check each count column against its own denominator.'
    },
    {
        'Check': 'Archetype share totals',
        'Expected': 'Each share column sums to 100%, within tolerance',
        'Actual': share_totals.round(4).to_dict('index'),
        'Pass/Flag': 'Pass' if checks.all() else 'Flag',
        'Notes': 'Tolerance: 0.1 percentage points.'
    },
    {
        'Check': 'Sector count totals',
        'Expected': 'Each total agrees with its corresponding archetype-table count',
        'Actual': sector_archetype_totals['Is_equal'].to_dict(),
        'Pass/Flag': (
            'Pass' if sector_archetype_totals['Is_equal'].all()
            else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'Unique master codes',
        'Expected': 'Zero duplicate SOC codes in Occupation_Master',
        'Actual': (
            'No duplicates' if occupation_master['SOC_code'].is_unique
            else f"{occupation_master['SOC_code'].duplicated().sum()} duplicates"
        ),
        'Pass/Flag': (
            'Pass' if occupation_master['SOC_code'].is_unique
            else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'Common thresholds',
        'Expected': 'Identical definitions, thresholds, transformations, and boundary rules',
        'Actual': 'Requires manual verification',
        'Pass/Flag': 'Flag',
        'Notes': 'Confirm shared classification logic across both samples.'
    },
    {
        'Check': 'Six occupation rows',
        'Expected': 'Exactly six requested occupations, with no duplicates',
        'Actual': f'{len(six_occupations_comparison_df)} rows',
        'Pass/Flag': (
            'Pass' if (
                len(six_occupations_comparison_df) == 6
                and not six_occupations_comparison_df.duplicated().any()
            ) else 'Flag'
        ),
        'Notes': ''
    },
    {
        'Check': 'Expansion run counts',
        'Expected': 'Expansion_runs <= Available_runs; all counted runs appear in the run key',
        'Actual': (
            'Inequality satisfied'
            if (
                six_occupations_comparison_df['Expansion_runs']
                <= six_occupations_comparison_df['Available_runs']
            ).all()
            else 'Inequality violated'
        ),
        'Pass/Flag': (
            'Pass' if (
                six_occupations_comparison_df['Expansion_runs']
                <= six_occupations_comparison_df['Available_runs']
            ).all()
            else 'Flag'
        ),
        'Notes': 'Run-key membership must be checked separately.'
    }
])

excel_tables['Checks'] = checks_df

# Make the excel File

In [103]:
from pathlib import Path

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

output_path = Path('../data/AI_Research_Technical_Outputs.xlsx')
output_path = Path('AI_Research_Technical_Outputs.xlsx')

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sheet_name, df in excel_tables.items():
        df.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False,
        )

        ws = writer.sheets[sheet_name[:31]]
        ws.freeze_panes = 'A2'

        # Format the header row.
        for cell in ws[1]:
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill(
                fill_type='solid',
                fgColor='1F4E78',
            )
            cell.alignment = Alignment(
                horizontal='left',
                vertical='center',
                wrap_text=True,
            )

        ws.row_dimensions[1].height = 32

        # Wrap cell contents and align values vertically.
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(
                    vertical='top',
                    wrap_text=True,
                )

        # Set readable column widths based on the actual contents.
        for column_idx, column_cells in enumerate(
            ws.iter_cols(),
            start=1,
        ):
            max_length = max(
                (
                    len(str(cell.value))
                    for cell in column_cells
                    if cell.value is not None
                ),
                default=0,
            )

            header = str(ws.cell(1, column_idx).value or '')
            width = min(max(max_length + 2, len(header) + 2), 45)

            ws.column_dimensions[
                get_column_letter(column_idx)
            ].width = width

        # Keep a final total row outside the filter range.
        last_row = ws.max_row
        filter_last_row = last_row

        if last_row > 1:
            first_value = str(
                ws.cell(last_row, 1).value or ''
            ).strip().lower()

            if first_value in {
                'total',
                'totals',
                'grand total',
                'overall total',
            }:
                filter_last_row -= 1

                for cell in ws[last_row]:
                    cell.font = Font(bold=True)
                    cell.fill = PatternFill(
                        fill_type='solid',
                        fgColor='D9EAF7',
                    )

        if filter_last_row >= 1 and ws.max_column > 0:
            ws.auto_filter.ref = (
                f'A1:{get_column_letter(ws.max_column)}'
                f'{filter_last_row}'
            )

print(f'Excel file saved to: {output_path.resolve()}')

Excel file saved to: /Users/jfv/Desktop/wadhwani/10_AI_research/04_creating_output/AI_Research_Technical_Outputs.xlsx
